# 02 · Edda indefra

## Opsætning

In [ ]:
import sys
from pathlib import Path

# VS Code runs a notebook in its own folder: walk up to training/, the folder with pyproject.toml.
TRAINING = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "pyproject.toml").is_file() and (p / "wt_train").is_dir())
sys.path.insert(0, str(TRAINING))

import logging
import warnings

# Two harmless start-up notices: no progress widgets, no torch.distributed redirects on macOS.
warnings.filterwarnings("ignore", message="IProgress not found")
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)

import matplotlib.pyplot as plt
import numpy as np
import torch
import transformers
from matplotlib.colors import PowerNorm
from transformers import WhisperForConditionalGeneration, WhisperProcessor

from wt_train import audio, fleurs, paths

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

## Edda og klippet

In [ ]:
EDDA = paths.snapshot_dir(paths.EDDA_REPO, paths.EDDA_REVISION)
processor = WhisperProcessor.from_pretrained(EDDA)
tokenizer = processor.tokenizer
# Eager attention hands back the attention weights; fp16 on the Apple GPU, as Edda ships.
model = WhisperForConditionalGeneration.from_pretrained(
    EDDA, dtype=torch.float16 if DEVICE == "mps" else torch.float32, attn_implementation="eager").to(DEVICE).eval()
encoder, decoder = model.model.encoder, model.model.decoder
for name, part in [("Koder", encoder), ("Afkoder", decoder)]:
    print(f"{name}: {len(part.layers)} lag, {sum(p.numel() for p in part.parameters()) / 1e6:.0f} mio. vægte")

clip = next(c for c in fleurs.load_split("test") if c.id == "fleurs-da_dk-1480530483950034476")
pcm = audio.read_16k_mono(clip.wav)
features = processor.feature_extractor(pcm, sampling_rate=audio.SAMPLE_RATE, return_tensors="pt").input_features
features = features.to(DEVICE, model.dtype)
print("Facit:", clip.text)
print("Ind i koderen:", tuple(features.shape), "på", DEVICE)

## Koderen

In [ ]:
with torch.inference_mode():
    encoded = encoder(features).last_hidden_state
SECONDS_PER_POSITION = processor.feature_extractor.chunk_length / encoded.shape[1]

print("Ud af koderen:", tuple(encoded.shape), "= klip × positioner × tal pr. position")
print("En position dækker", round(1000 * SECONDS_PER_POSITION), "ms lyd")

## Ét token ad gangen

In [ ]:
PREFIX = tokenizer.convert_tokens_to_ids(["<|startoftranscript|>", "<|da|>", "<|transcribe|>", "<|notimestamps|>"])
EOT = tokenizer.convert_tokens_to_ids("<|endoftext|>")


def show(token):
    return repr(tokenizer.decode([token]))  # the quotes show a word's leading space


tokens, sure, listened = PREFIX[:1], [], []
print(f"{'trin':>4}  {'':7}  {'token':<18}  de 5 mest sandsynlige næste tokens")
with torch.inference_mode():
    while tokens[-1] != EOT and len(tokens) < 100:
        out = model(encoder_outputs=(encoded,), decoder_input_ids=torch.tensor([tokens], device=DEVICE),
                    output_attentions=True)
        probs = out.logits[0, -1].float().softmax(-1)  # the next token's probabilities
        forced = len(tokens) < len(PREFIX)
        token = PREFIX[len(tokens)] if forced else int(probs.argmax())  # greedy: the most likely
        top = probs.topk(5)
        print(f"{len(tokens):>4}  {'tvunget' if forced else 'valgt':7}  {show(token):<18} ",
              "  ".join(f"{show(int(i))} {p:.1%}" for p, i in zip(top.values, top.indices)))
        sure.append(float(probs[token]))
        # Cross-attention of the newest position: layers × heads × 1500 encoder positions.
        listened.append(torch.stack(out.cross_attentions)[:, 0, :, -1].float().cpu().numpy())
        tokens.append(token)
    generated = model.generate(input_features=features, language="da", task="transcribe", do_sample=False,
                               num_beams=1, return_timestamps=False)[0].tolist()

print("\nEdda skrev:", tokenizer.decode(tokens, skip_special_tokens=True))
print("Samme tekst-tokens som model.generate:", generated == tokens[len(PREFIX):-1])

## Krydsopmærksomhed: hvor Edda lytter

In [ ]:
step = min(range(len(PREFIX) - 1, len(sure)), key=sure.__getitem__)  # the step Edda was least sure of
layers, heads, positions = listened[step].shape
end = len(pcm) / audio.SAMPLE_RATE

window = positions * SECONDS_PER_POSITION
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True, layout="constrained", width_ratios=[3, 2])
for ax, right in zip(axes, [window, end]):
    im = ax.imshow(listened[step].reshape(layers * heads, positions), aspect="auto", cmap="magma",
                   norm=PowerNorm(0.5), interpolation="nearest", extent=(0, window, layers * heads, 0))
    ax.hlines(np.arange(1, layers) * heads, 0, window, color="white", linewidth=0.8)
    ax.axvline(end, color="cyan", linestyle="--", label=f"optagelsen slutter ({end:.1f} s)".replace(".", ","))
    ax.set(xlim=(0, right), xlabel="Tid i lyden (sekunder)")
axes[0].set_yticks(np.arange(layers) * heads + heads / 2, [f"lag {i + 1}\n{heads} hoveder" for i in range(layers)])
axes[0].set_title(f"Alle {positions} positioner ({window:.0f} sekunder)")
axes[0].legend(loc="upper center")
axes[1].set_title("Kun optagelsen")
fig.suptitle(f"Trin {step + 1}: hvor Edda lyttede, da den valgte {show(tokens[step + 1])} ({sure[step]:.0%} sikker)")
fig.colorbar(im, ax=axes, label="Opmærksomhed (vægt)")
plt.show()

## Hvert token på tidslinjen

In [ ]:
# The heads Whisper times words with (Edda's generation config), averaged.
ALIGNMENT_HEADS = model.generation_config.alignment_heads
text_steps = range(len(PREFIX) - 1, len(tokens) - 1)
timeline = np.array([np.mean([listened[s][layer, head] for layer, head in ALIGNMENT_HEADS], axis=0)
                     for s in text_steps])
labels = [show(tokens[s + 1]) for s in text_steps]

fig, ax = plt.subplots(figsize=(12, 8), layout="constrained")
ax.imshow(timeline, aspect="auto", cmap="magma", norm=PowerNorm(0.5), interpolation="nearest",
          extent=(0, positions * SECONDS_PER_POSITION, len(labels), 0))
ax.set_yticks(np.arange(len(labels)) + 0.5, labels)
ax.axvline(end, color="cyan", linestyle="--", label="optagelsen slutter")
ax.set(xlim=(0, end + 1), title=f"Hvor hvert token lytter ({len(ALIGNMENT_HEADS)} justeringshoveder i gennemsnit)",
       xlabel="Tid i lyden (sekunder)")
ax.legend(loc="upper right")
plt.show()